# Session 3-4: Building a Chatbot with an LLM API

**Learning goals:**

- Call Gemini and OpenRouter APIs.
- Build a multi-turn chatbot.
- Use Function Calling to answer weather questions and explain the link to MCP.

**Prerequisites:**

- Complete Sessions 1 and 2; use Python with `google-genai` and `requests` installed.
- Have an LLM API key and model ID, plus a WeatherAPI key for the tool activity.

## 1. LLM API

An LLM API lets a Python application send text to a language model and receive generated text. In this session, the input is a **prompt** and the output is a **text response**. A prompt gives the model an instruction, such as translating a sentence or answering a question.

![LLM API structure: a text prompt is sent in a JSON request to an endpoint using an API key and model ID; generated text is extracted from the JSON response.](assets/session3/llm-api-structure.svg)

Python sends the text in a JSON request over HTTP. The API returns a JSON response, from which Python extracts the generated text. Later, Function Calling adds structured tool requests to this flow.

### Necessary items to call an LLM API?

| Item | Purpose |
|---|---|
| API endpoint | The URL that receives the HTTP request |
| API key | Authenticates access to the service |
| Model ID | Selects the language model |
| Input text | The prompt or conversation sent to the model |

Send a `POST` request with authentication headers and a JSON body. An optional system instruction can guide the model's behaviour. Read the response JSON to find the generated text.

- [Gemini API documentation](https://ai.google.dev/gemini-api/docs/generate-content/text-generation) · [Get a Gemini API key](https://aistudio.google.com/) · [Gemini models](https://ai.google.dev/gemini-api/docs/models)
- [OpenRouter API documentation](https://openrouter.ai/docs/api/api-reference/chat/send-chat-completion-request) · [Get an OpenRouter API key](https://openrouter.ai/keys) · [OpenRouter models](https://openrouter.ai/models)

A **token** is a chunk of text processed by the model. Longer prompts and conversations use more tokens. Generated answers can be incorrect.

## 2. Call an LLM API

Replace the key placeholders with your own keys before running the examples. Install the libraries in your active terminal:

```bash
python -m pip install google-genai requests
```

### 2.1 Gemini API

Call Gemini using either the SDK or a direct REST request. Both examples send `Hi` and print the generated text.

The example uses `gemini-3.8-flash`. A [free tier](https://ai.google.dev/gemini-api/docs/pricing) is available subject to account access and quotas.

The [Google GenAI SDK](https://ai.google.dev/gemini-api/docs/libraries) handles HTTP and JSON for you.

In [ ]:
from google import genai

GEMINI_API_KEY = "YOUR_GEMINI_API_KEY"  # Replace with your actual Gemini API key.

# Create an authenticated client for the Gemini API.
gemini_client = genai.Client(api_key=GEMINI_API_KEY)
GEMINI_MODEL = "gemini-3.8-flash"

response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,  # Select the language model.
    contents="Hi",  # Text prompt sent to the model.
)
print(response.text)

Hello! How can I help you today?


Define the chat endpoint, API key and model, then send a `POST` request with `requests`. Gemini's compatible chat endpoint accepts a `messages` list and returns the generated text in `choices`.


[Gemini compatible REST API documentation](https://ai.google.dev/gemini-api/docs/openai)

In [ ]:
import requests

CHAT_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
CHAT_API_KEY = "YOUR_GEMINI_API_KEY"  # Replace with your actual Gemini API key.
CHAT_MODEL = "gemini-3.8-flash"

response = requests.post(
    CHAT_URL,
    headers={"Authorization": f"Bearer {CHAT_API_KEY}"},  # Authenticate access to Gemini.
    json={
        "model": CHAT_MODEL,
        "messages": [{"role": "user", "content": "Hi"}],
    },  # Send the model and input message as JSON.
    timeout=60,  # Maximum wait for the API response, in seconds.
)
response.raise_for_status()  # Stop if the server returns an HTTP error.
print(response.json()["choices"][0]["message"]["content"])


### 2.2 OpenRouter API

Send a message with `requests.post()` and print the answer. OpenRouter uses a `messages` list containing the user's text. This model ID selects Gemini through OpenRouter; you can choose another [available model](https://openrouter.ai/models?supported_parameters=tools). The weather chatbot later needs a model that supports tools.

In [ ]:
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

OPENROUTER_API_KEY = "YOUR_OPENROUTER_API_KEY"  # Replace with your actual OpenRouter API key.
OPENROUTER_MODEL = "minimax/minimax-m3.1-flash:free"

response = requests.post(
    OPENROUTER_URL,
    headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"},  # Authenticate access.
    json={
        "model": OPENROUTER_MODEL,
        "messages": [{"role": "user", "content": "Hi"}],
    },  # Send the model ID and user message as JSON.
    timeout=60,  # Maximum wait for the API response, in seconds.
)
response.raise_for_status()  # Stop if the server returns an HTTP error.
print(response.json()["choices"][0]["message"]["content"])


Hi! How can I help you today? 😊


### 2.3 Error codes and troubleshooting

| Status | Meaning and next check |
|---|---|
| 400 or 422 | Invalid request: check fields and model capabilities |
| 401 or 403 | Authentication or access problem: check your key and permissions |
| 402 | Check account credit or billing requirements |
| 404 | Check the endpoint and exact model ID |
| 429 | Quota or rate limit reached: reduce calls or try later |
| 5xx | Server error: check service availability and try later |

A timeout means the request did not finish within the allowed wait. An SDK may report these failures as exceptions.

## 3. Build a chatbot

### 3.1 Use a system prompt

Keep messages in a `conversation` list. Each message has a `role` and `content`:

- `system`: the assistant's identity and task, defined by `system_prompt`.
- `user`: ordinary input such as `Bonjour` or a paragraph.
- `assistant`: the model's response.

Here the system prompt makes the assistant a translator into Australian English. The user supplies only the text to translate.

We use [Gemini's compatible chat endpoint](https://ai.google.dev/gemini-api/docs/openai), which accepts the same message structure as OpenRouter.

In [ ]:
CHAT_URL = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
CHAT_API_KEY = "YOUR_GEMINI_API_KEY"  # Replace with your actual Gemini API key.
CHAT_MODEL = "gemini-3.8-flash"

# To use OpenRouter, replace the above `CHAT_URL`, `CHAT_API_KEY`, and `CHAT_MODEL` with the following:
CHAT_URL = "https://openrouter.ai/api/v1/chat/completions"
CHAT_API_KEY = "YOUR_OPENROUTER_API_KEY"  # Replace with your actual OpenRouter API key.
CHAT_MODEL = "minimax/minimax-m3.1-flash:free"


def send_messages(messages):
    """Send a message list and return the complete assistant message."""
    response = requests.post(
        CHAT_URL,
        headers={"Authorization": f"Bearer {CHAT_API_KEY}"},  # Authenticate with the selected service.
        json={"model": CHAT_MODEL, "messages": messages},  # Send the model and conversation.
        timeout=60,  # Maximum wait for the response, in seconds.
    )
    response.raise_for_status()
    reply = response.json()["choices"][0]["message"]
    if not isinstance(reply.get("content"), str) or not reply["content"].strip():
        raise RuntimeError("The model did not return a text answer.")
    return reply


system_prompt = (
    "You are a good translator. Translate the user's input into Australian English. "
    "Return only the translation."
)
conversation = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Bonjour"},
]
assistant_message = send_messages(conversation)
conversation.append(assistant_message)  # Keep the reply with its assistant role.
print(assistant_message["content"])

### 3.2 Keep a conversation history

For this text conversation, we use three roles:

- `system`: the assistant's identity and task.
- `user`: a message entered by the person.
- `assistant`: the answer generated by the model.

The `conversation` list now holds the system prompt, the user's input and the assistant's translation. Send that history with each new input, then append the new user message and assistant answer. Earlier passages can provide context and help maintain consistent terminology.

The `send_messages()` helper returns the model's message as a dictionary. Read its `content` into `assistant_msg`, then save it with `role: "assistant"`. This REST example uses `response["content"]`; the Gemini SDK example in Section 2.1.1 uses `response.text` instead.

The function below works with either Gemini's compatible REST endpoint or OpenRouter. It saves both messages after the request succeeds.

In [ ]:
system_prompt = (
    "You are a good translator. Translate the user's input into Australian English. "
    "Return only the translation."
)
conversation = [
    {"role": "system", "content": system_prompt},
]
def chat(user_message: str) -> str:
    """Translate a new message and save the user and assistant messages."""
    user_msg = {"role": "user", "content": user_message}
    response = send_messages(conversation + [user_msg])  # Send the history and new input.
    assistant_msg = response["content"]  # Extract the model's answer text.

    conversation.append(user_msg)
    conversation.append({"role": "assistant", "content": assistant_msg})
    return assistant_msg


print(chat("Je voudrais réserver une table pour deux personnes."))


### 3.3 Interactive translator: type quit to leave

Run the next cell and enter `Bonjour`, another sentence or a paragraph. The system prompt already defines the task, so there is no need to type “translate” with each input.

The same `chat()` function works with either API. Blank input is skipped, and `quit` exits before sending a request. Rerunning the cell resets the conversation to its system prompt.

In [ ]:
# Start a fresh conversation while keeping the translator's system prompt.
system_prompt = (
    "You are a good translator. Translate the user's input into Australian English. "
    "Return only the translation."
)
conversation = [{"role": "system", "content": system_prompt}]
print("Translator started. Type quit to finish.")

while True:
    user_content = input("You: ").strip()
    if user_content.lower() == "quit":
        print("Translator ended.")
        break
    if not user_content:
        continue
    print("Translation:", chat(user_content))


## 4. Function Calling: answer weather questions

### 4.1 How Function Calling works

**Tool calling at run time**

The LLM requests a tool; Python runs it and returns the result with the matching call ID. The LLM then answers the user.

![Tool calling cycle: user question, LLM tool request, Python execution, weather data, tool result and final answer.](assets/session3/function-calling-cycle.svg)

**Python implementation workflow: Sections 4.2–4.5**

Define the Python function, describe it in `tools`, connect its name through a mapping, then call it from `chat()`.

![Python implementation workflow: function definition, tool definition, tool mapping and calling the tool in chat.](assets/session3/tools-chat-workflow.svg)


### 4.2 Defining a function

Reuse Session 2's `get_weather(city)` function. It returns the city, temperature and condition. Replace `YOUR_WEATHERAPI_KEY` with your [WeatherAPI key](https://www.weatherapi.com/), then test the function.

In [ ]:
WEATHER_API_KEY = "YOUR_WEATHERAPI_KEY"
CURRENT_WEATHER_URL = "https://api.weatherapi.com/v1/current.json"


def get_weather(city: str):
    """Return simplified weather for a city."""
    # key authenticates the request; q selects a location; aqi omits air-quality data.
    params = {"key": WEATHER_API_KEY, "q": city.strip(), "aqi": "no"}
    response = requests.get(
        CURRENT_WEATHER_URL,
        params=params,
        timeout=10,  # Maximum wait for the weather response, in seconds.
    )
    response.raise_for_status()
    data = response.json()

    # Flatten the location and current objects into one small record.
    return {
        "city": data["location"]["name"],
        "current_temp": data["current"]["temp_c"],
        "condition": data["current"]["condition"]["text"],
    }


print(get_weather("Perth"))


### 4.3 Defining tools

Each tool has a `name`, a `description` and an `input_schema`. The schema lists its arguments and their types; `required` lists mandatory arguments.

We keep this simple structure in Python. The request helper in Section 4.5 converts it into the format required by the selected API.


In [ ]:
import json

tools = [
    {
        "name": "get_weather",
        "description": "Get current weather for a city in degrees Celsius. Not a forecast.",
        "input_schema": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, such as Perth, Australia.",
                }
            },
            "required": ["city"],
            "additionalProperties": False,
        },
    }
]


### 4.4 Mapping tools to functions

The mapping connects a tool name to its Python function. `**tool_args` passes the argument dictionary as keyword arguments: `{"city": "Perth"}` becomes `city="Perth"`.

In [ ]:
mapping_tool_function = {
    "get_weather": get_weather,
}


def execute_tool(tool_name, tool_args):
    """Run the mapped Python function with the model's proposed arguments."""
    return mapping_tool_function[tool_name](**tool_args)

### 4.5 Calling tools in chat

Reuse Section 3's request helper and add the tool definitions. Gemini's compatible endpoint and OpenRouter require a `function` wrapper with a `parameters` field. Convert `input_schema` to `parameters` when building the request; the Python tool definitions and mapping stay the same.


In [ ]:
def send_messages(messages, tools=None, tool_choice="auto"):
    """Send a conversation with optional tools to the selected API."""
    payload = {"model": CHAT_MODEL, "messages": messages}
    if tools is not None:
        api_tools = [
            {
                "type": "function",
                "function": {
                    "name": tool["name"],
                    "description": tool["description"],
                    "parameters": tool["input_schema"],  # Use the API's field name.
                },
            }
            for tool in tools
        ]
        payload.update(tools=api_tools, tool_choice=tool_choice)
    response = requests.post(
        CHAT_URL,
        headers={"Authorization": f"Bearer {CHAT_API_KEY}"},
        json=payload,
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]

Extend `chat()` to run any requested tools and add their results with `role: "tool"`. The call ID links each result to its request. Then request the final answer with `tool_choice="none"`.

In [ ]:
def chat(user_message: str) -> str:
    """Complete a chat turn, running any requested weather tools."""
    pending = conversation + [{"role": "user", "content": user_message}]
    response = send_messages(pending, tools=tools)
    pending.append(response)  # Preserve the full reply, including tool calls and their IDs.

    if response.get("tool_calls"):
        for call in response["tool_calls"]:
            tool_name = call["function"]["name"]
            tool_args = json.loads(call["function"]["arguments"])  # Read the JSON arguments.
            result = execute_tool(tool_name, tool_args)  # Run the mapped Python function.
            pending.append({
                "role": "tool",
                "tool_call_id": call["id"],
                "content": json.dumps(result),
            })
        response = send_messages(pending, tools=tools, tool_choice="none")
        pending.append(response)

    conversation[:] = pending  # Save the complete turn after the requests succeed.
    return response["content"]

Use the same input loop as Section 3, with a weather system prompt. Try “What is the weather in Perth?”, then “What about Sydney?”. Type `quit` to finish.

In [ ]:
system_prompt = (
    "You are a friendly weather assistant. Use Australian English. "
    "Use get_weather for current observations and do not invent weather data. "
    "Ask for a city when it is missing. This tool does not provide forecasts."
)
conversation = [{"role": "system", "content": system_prompt}]
print("Weather chat started. Type quit to finish.")

while True:
    user_content = input("You: ").strip()
    if user_content.lower() == "quit":
        print("Weather chat ended.")
        break
    if not user_content:
        continue
    print("Assistant:", chat(user_content))

### 4.6 Download and practise

**Weather chatbot**

[Download the complete Python chatbot](assets/session3/weather_chatbot.py)

Save the file, enter your LLM and WeatherAPI keys, then run it in your terminal:

```bash
python -m pip install requests
python weather_chatbot.py
```

**Multiple-tools practice notebook**

[Open the multiple-tools practice notebook](<Session 3-4 Practice Chatbot with Multiple Tools.ipynb>)

Extend the chatbot with current weather, forecasts, historical weather and arXiv searches.


### 4.7 Why introduce MCP?

Function Calling lets the LLM select a tool and propose arguments. Our application still defines and maps each tool in code. MCP provides a shared protocol for connecting applications to tool servers.

MCP helps us:

- Discover available tools through `tools/list`.
- Call tools through a standard `tools/call` request.
- Describe tool inputs with JSON Schema using `inputSchema`.
- Reuse a tool server across different AI applications.
- Keep tool implementations separate from chatbot code.
